# BERT vs LLM Sentiment Analysis

> **Research framing:** Small-scale, controlled experiment.
> The goal is **understanding**, not performance — not benchmark chasing.

This notebook compares two approaches to binary sentiment classification:

| Approach | Model | Resource class |
|----------|-------|---------------|
| Pretrained classifier | `distilbert-base-uncased-finetuned-sst-2-english` | Low-resource |
| Zero-shot LLM | `gpt-4o-mini` | High-resource |

**Dataset:** IMDb reviews — 500 samples (250 pos / 250 neg), fixed seed = 42.

---

## Section 0 — Setup & Hypothesis

Load dependencies, configure constants, and pre-register hypotheses *before* running
a single line of inference. Pre-registering keeps us honest — we can't explain
results we observed before we looked at the data.

In [1]:
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise EnvironmentError(
        'OPENAI_API_KEY not found. '
        'Copy .env.example to .env and fill in your key.'
    )

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Config ───────────────────────────────────────────────────────────────
SAMPLE_SIZE    = 500   # 250 positive + 250 negative
LONG_THRESHOLD = 40    # words — threshold set HERE, before any labeling
LLM_MODEL      = 'gpt-4o-mini'

print('Setup complete.')
print(f'LLM: {LLM_MODEL} | Sample size: {SAMPLE_SIZE} | '
      f'Long threshold: {LONG_THRESHOLD} words')

Setup complete.
LLM: gpt-4o-mini | Sample size: 500 | Long threshold: 40 words


### Pre-Registered Hypotheses

Defined *before* running experiments. These guide analysis — not explain it post-hoc.

| # | Hypothesis |
|---|------------|
| **H1** | LLM performs better on **mixed sentiment and long-context reviews**; BERT performance degrades when sentiment is not locally expressed |
| **H2** | BERT is **faster and cheaper** but brittle on sarcasm and long context |
| **H3** | LLM handles **long reviews** better due to larger context window |
| **H4** | On **clear-cut positive/negative** samples, both models are roughly equivalent |

---

## Section 1 — Dataset

Source: [IMDb via HuggingFace](https://huggingface.co/datasets/imdb) — English-only movie reviews.

- **500 samples** (250 positive / 250 negative, stratified)
- Fixed `random_state=42` for reproducibility
- Saved to `data/sample.csv` — downstream analysis doesn't need re-downloading

In [2]:
from datasets import load_dataset

print('Loading IMDb from HuggingFace (test split)...')
dataset = load_dataset('imdb', split='test')  # 25k samples
df = dataset.to_pandas()

# Stratified sample: equal class balance
pos = df[df['label'] == 1].sample(n=SAMPLE_SIZE // 2, random_state=SEED)
neg = df[df['label'] == 0].sample(n=SAMPLE_SIZE // 2, random_state=SEED)
sample_df = (
    pd.concat([pos, neg])
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

# Word count — used later for 'long' tagging
sample_df['word_count'] = sample_df['text'].str.split().str.len()

Path('data').mkdir(exist_ok=True)
sample_df.to_csv('data/sample.csv', index=False)

n_pos = sample_df['label'].sum()
n_neg = (sample_df['label'] == 0).sum()
n_long = (sample_df['word_count'] > LONG_THRESHOLD).sum()
print(f'Loaded: {len(sample_df)} samples | {n_pos} positive | {n_neg} negative')
print(f'Avg word count: {sample_df["word_count"].mean():.0f} '
      f'| Max: {sample_df["word_count"].max()} '
      f'| Min: {sample_df["word_count"].min()}')
print(f'Long reviews (>{LONG_THRESHOLD} words): {n_long} '
      f'({n_long/len(sample_df):.1%})')
print()
sample_df[['text', 'label', 'word_count']].head()

Loading IMDb from HuggingFace (test split)...


Loaded: 500 samples | 250 positive | 250 negative
Avg word count: 236 | Max: 1088 | Min: 24
Long reviews (>40 words): 496 (99.2%)



,text,label,word_count
0,I'd have to say that I've seen worse Sci Fi Ch...,0,460
1,"This film grabbed me right from its start, whe...",1,1001
2,With Pep Squad receiving an average of 4.7 on ...,0,257
3,I just love this movie and I have my TV progra...,1,167
4,This film certainly wasn't very sophisticated....,1,150


## Section 2 — BERT Baseline

Model: `distilbert-base-uncased-finetuned-sst-2-english`

- Already fine-tuned on SST-2 (Stanford Sentiment Treebank)
- **No training here** — using it as a pretrained classifier only
- Truncates at 512 tokens (~350 words). Reviews longer than that get cut off.
- Runs on CPU — no GPU required

This is the *low-resource* baseline: cheap, fast, widely deployed.

In [3]:
from transformers import pipeline

print('Loading distilbert pipeline (first run downloads ~250MB)...')
bert = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512,
    device=-1,  # CPU
)
print('BERT pipeline ready.')

Loading distilbert pipeline (first run downloads ~250MB)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

BERT pipeline ready.


In [4]:
from tqdm import tqdm
bert_results = []

for text in tqdm(sample_df['text'].tolist(), desc='BERT inference'):
    t0 = time.perf_counter()
    out = bert(text)[0]
    t1 = time.perf_counter()
    bert_results.append({
        'bert_pred':       1 if out['label'] == 'POSITIVE' else 0,
        'bert_label_raw':  out['label'],
        'bert_confidence': round(out['score'], 4),
        'bert_latency_s':  round(t1 - t0, 4),
    })

bert_df = pd.DataFrame(bert_results)
bert_df['bert_correct'] = (bert_df['bert_pred'] == sample_df['label'].values)
print('BERT inference complete.')

BERT inference: 100%|██████████| 500/500 [01:10<00:00,  7.12it/s]

BERT inference complete.


In [5]:
bert_accuracy    = bert_df['bert_correct'].mean()
bert_avg_latency = bert_df['bert_latency_s'].mean()
bert_avg_conf    = bert_df['bert_confidence'].mean()

# Confidence on correct vs incorrect — key calibration signal
right_conf = bert_df[ bert_df['bert_correct']]['bert_confidence'].mean()
wrong_conf = bert_df[~bert_df['bert_correct']]['bert_confidence'].mean()

print('=== BERT Results ===')
print(f'Accuracy:                    {bert_accuracy:.1%}')
print(f'Avg latency:                 {bert_avg_latency * 1000:.1f} ms/sample')
print(f'Avg confidence:              {bert_avg_conf:.3f}')
print(f'Confidence on CORRECT:       {right_conf:.3f}')
print(f'Confidence on INCORRECT:     {wrong_conf:.3f}  <- watch this gap')

=== BERT Results ===
Accuracy:                    88.2%
Avg latency:                 139.9 ms/sample
Avg confidence:              0.974
Confidence on CORRECT:       0.985
Confidence on INCORRECT:     0.899  <- watch this gap


## Section 3 — LLM Zero-Shot (`gpt-4o-mini`)

**Prompt design:**
- System: roles the model as a binary classifier
- User: review text + explicit output constraint
- **Forced output format:** `Answer ONLY with: positive or negative`
- Temperature = 0 (deterministic)
- `max_tokens = 5` (cost control; forces concise response)
- Input truncated to **first 200 words** (known limitation — see Section 6)

No few-shot examples. This is pure zero-shot.

In [6]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = 'You are a binary sentiment classifier. Be concise and precise.'


def build_prompt(text: str) -> str:
    # Truncate to 200 words — cost control.
    # NOTE: this is a known limitation. Reviews where sentiment resolves
    # after word 200 will likely be mislabeled.
    words = text.split()
    truncated = ' '.join(words[:200]) if len(words) > 200 else text
    return (
        f'Classify the sentiment of this movie review:\n\n'
        f'{truncated}\n\n'
        f'Answer ONLY with: positive or negative'
    )


def classify_with_llm(text: str) -> dict:
    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_prompt(text)},
        ],
        max_tokens=5,
        temperature=0,
    )
    t1 = time.perf_counter()
    raw  = response.choices[0].message.content.strip().lower()
    pred = 1 if 'positive' in raw else 0
    return {
        'llm_pred':          pred,
        'llm_label_raw':     raw,
        'llm_latency_s':     round(t1 - t0, 4),
        'llm_input_tokens':  response.usage.prompt_tokens,
        'llm_output_tokens': response.usage.completion_tokens,
    }


print('LLM inference functions ready.')
print(f'Model: {LLM_MODEL}')

LLM inference functions ready.
Model: gpt-4o-mini


In [7]:
# ⚠️  This cell makes real API calls to OpenAI.
# Est. cost: ~500 reviews × ~150 tokens × $0.15/1M input ≈ $0.01–0.04 total

llm_results = []

for text in tqdm(sample_df['text'].tolist(), desc='LLM inference'):
    try:
        result = classify_with_llm(text)
    except Exception as e:
        result = {
            'llm_pred':          -1,
            'llm_label_raw':     f'ERROR: {e}',
            'llm_latency_s':     0.0,
            'llm_input_tokens':  0,
            'llm_output_tokens': 0,
        }
    llm_results.append(result)
    time.sleep(0.05)  # gentle rate limiting

llm_df = pd.DataFrame(llm_results)
llm_df['llm_correct'] = (llm_df['llm_pred'] == sample_df['label'].values)

n_errors = (llm_df['llm_pred'] == -1).sum()
print(f'LLM inference complete. Errors: {n_errors}')

LLM inference: 100%|██████████| 500/500 [09:15<00:00,  1.11s/it]

LLM inference complete. Errors: 0


In [8]:
llm_accuracy    = llm_df['llm_correct'].mean()
llm_avg_latency = llm_df['llm_latency_s'].mean()
total_input     = llm_df['llm_input_tokens'].sum()
total_output    = llm_df['llm_output_tokens'].sum()

# gpt-4o-mini pricing (as of Q1 2025)
# $0.15 / 1M input tokens | $0.60 / 1M output tokens
cost_input  = total_input  / 1_000_000 * 0.15
cost_output = total_output / 1_000_000 * 0.60
total_cost  = cost_input + cost_output

print('=== LLM Results ===')
print(f'Accuracy:                {llm_accuracy:.1%}')
print(f'Avg latency:             {llm_avg_latency * 1000:.0f} ms/sample')
print(f'Total tokens:            {total_input:,} input / {total_output:,} output')
print(f'Total cost ({SAMPLE_SIZE} samples): ${total_cost:.4f}')
print(f'Est. cost / 1k samples:  ${total_cost / SAMPLE_SIZE * 1000:.3f}')

=== LLM Results ===
Accuracy:                94.8%
Avg latency:             1061 ms/sample
Total tokens:            119,594 input / 500 output
Total cost (500 samples): $0.0182
Est. cost / 1k samples:  $0.036


## Section 4 — Comparison

Merging both result sets. The summary table gives a rough snapshot —
the **disagreement cases** (where BERT and LLM differ) are the more
interesting signal.

In [9]:
# Merge into one results DataFrame
results_df = (
    sample_df[['text', 'label', 'word_count']]
    .copy()
    .rename(columns={'label': 'ground_truth'})
)
results_df = pd.concat([results_df, bert_df, llm_df], axis=1)

# Pre-define category tags (long is automatic; mixed/sarcasm annotated in Sec 5)
results_df['tag_long']    = results_df['word_count'] > LONG_THRESHOLD
results_df['tag_mixed']   = False  # manually identified in Section 5
results_df['tag_sarcasm'] = False  # manually identified in Section 5

results_df.to_csv('data/results.csv', index=False)
print('Saved to data/results.csv')
print()

# Summary
comparison = pd.DataFrame({
    'Model':                  ['BERT (distilbert)', f'LLM ({LLM_MODEL})'],
    'Accuracy':               [f'{bert_accuracy:.1%}', f'{llm_accuracy:.1%}'],
    'Avg Latency':            [f'{bert_avg_latency * 1000:.1f} ms', f'{llm_avg_latency * 1000:.0f} ms'],
    'Est. Cost / 1k samples': ['~$0.00 (local CPU)', f'${total_cost / SAMPLE_SIZE * 1000:.3f}'],
})
display(comparison)

Saved to data/results.csv



,Model,Accuracy,Avg Latency,Est. Cost / 1k samples
0,BERT (distilbert),88.2%,139.9 ms,~$0.00 (local CPU)
1,LLM (gpt-4o-mini),94.8%,1061 ms,$0.036


In [10]:
disagree = results_df[results_df['bert_pred'] != results_df['llm_pred']].copy()
print(f'Disagreements: {len(disagree)} / {len(results_df)} '
      f'({len(disagree) / len(results_df):.1%})')
print()

sample_disagree = disagree.sample(min(10, len(disagree)), random_state=SEED)

for _, row in sample_disagree.iterrows():
    gt = 'POS' if row['ground_truth'] == 1 else 'NEG'
    bp = 'POS' if row['bert_pred']    == 1 else 'NEG'
    lp = 'POS' if row['llm_pred']     == 1 else 'NEG'
    bm = 'correct' if row['bert_correct'] else 'WRONG '
    lm = 'correct' if row['llm_correct']  else 'WRONG '
    tag = '[LONG]' if row['tag_long'] else ''
    print(f'[GT:{gt}]  BERT:{bp} ({bm})  LLM:{lp} ({lm})  {tag}')
    print(f'  {row["text"][:250]}...')
    print()

Disagreements: 53 / 500 (10.6%)

[GT:POS]  BERT:NEG (WRONG )  LLM:POS (correct)  [LONG]
  "Who Will Love My Children" Saddest movie I have ever seen. Definite 10/10. Released on TV in 1983. Movie has been released on VHS. DVD release is a must, sooner rather than later. Mother dying of cancer, must find homes for all her children before s...

[GT:POS]  BERT:NEG (WRONG )  LLM:POS (correct)  [LONG]
  The first time i saw this movie was on a flight between Guangzhou, China and Los Angeles. It was a real hoot and made the trip pass with much less discomfort than the normal 10 hour flight. I tried to locate a copy of it without success until I disco...

[GT:NEG]  BERT:POS (WRONG )  LLM:NEG (correct)  [LONG]
  Hulk Hogan stars as a champion wrestler (A real acting stretch...) named Rip, who is forced to defend his honor, his title and his girlfriend from a greedy corporation that wanted him to sign for their network (Because wrestling sells!) however when ...

[GT:POS]  BERT:NEG (WRONG )  LL

## Section 5 — Failure Case Analysis ⭐

> This is the core section. Accuracy numbers are a summary — failure cases tell you *why*.

### Tagging Schema

| Tag | Definition |
|-----|------------|
| `mixed` | Review contains both positive **and** negative clauses in the same text |
| `long` | Word count **> 40** — threshold set before labeling (no confirmation bias) |
| `sarcasm` | Sentiment is inverted from literal word meaning (manual; rare) |
| `ambiguous` | Neither clearly positive nor negative, even to a human reader |
| `clear` | Unambiguous polarity, straightforward vocabulary |

In [11]:
bert_fails      = results_df[~results_df['bert_correct']]
llm_fails       = results_df[~results_df['llm_correct']]
both_fail       = results_df[~results_df['bert_correct'] & ~results_df['llm_correct']]
only_bert_fails = results_df[~results_df['bert_correct'] &  results_df['llm_correct']]
only_llm_fails  = results_df[ results_df['bert_correct'] & ~results_df['llm_correct']]

print('=== Failure Overview ===')
print(f'BERT failures:    {len(bert_fails):3d} ({len(bert_fails) / len(results_df):.1%})')
print(f'LLM failures:     {len(llm_fails):3d} ({len(llm_fails) / len(results_df):.1%})')
print(f'Both fail:        {len(both_fail):3d} ({len(both_fail) / len(results_df):.1%})')
print(f'Only BERT fails:  {len(only_bert_fails):3d} ({len(only_bert_fails) / len(results_df):.1%})  <- LLM-exclusive wins')
print(f'Only LLM fails:   {len(only_llm_fails):3d} ({len(only_llm_fails) / len(results_df):.1%})  <- BERT-exclusive wins')
print()

# Long vs short breakdown
long_df  = results_df[ results_df['tag_long']]
short_df = results_df[~results_df['tag_long']]

print(f'=== Long vs Short Reviews ===')
print(f'Long  (>{LONG_THRESHOLD} words): {len(long_df):3d} samples '
      f'| BERT: {long_df["bert_correct"].mean():.1%} '
      f'| LLM: {long_df["llm_correct"].mean():.1%}')
print(f'Short (<={LONG_THRESHOLD} words): {len(short_df):3d} samples '
      f'| BERT: {short_df["bert_correct"].mean():.1%} '
      f'| LLM: {short_df["llm_correct"].mean():.1%}')

=== Failure Overview ===
BERT failures:     59 (11.8%)
LLM failures:      26 (5.2%)
Both fail:         16 (3.2%)
Only BERT fails:   43 (8.6%)  <- LLM-exclusive wins
Only LLM fails:    10 (2.0%)  <- BERT-exclusive wins

=== Long vs Short Reviews ===
Long  (>40 words): 496 samples | BERT: 88.1% | LLM: 94.8%
Short (<=40 words):   4 samples | BERT: 100.0% | LLM: 100.0%


In [12]:
print('=== BERT FAILURE CASES (LLM was correct on these) ===')
print('i.e., cases the LLM handles that BERT cannot\n')

sample_bert_fails = only_bert_fails.sample(
    min(8, len(only_bert_fails)), random_state=SEED
)

for i, (_, row) in enumerate(sample_bert_fails.iterrows()):
    gt_str   = 'positive' if row['ground_truth'] == 1 else 'negative'
    bert_str = 'positive' if row['bert_pred']    == 1 else 'negative'
    tag_str  = '[LONG]' if row['tag_long'] else ''
    print(f'[{i+1}] Truth: {gt_str} | BERT: {bert_str} '
          f'| Confidence: {row["bert_confidence"]:.3f} '
          f'| Words: {row["word_count"]} {tag_str}')
    print(f'     {row["text"][:400]}...')
    print()

=== BERT FAILURE CASES (LLM was correct on these) ===
i.e., cases the LLM handles that BERT cannot

[1] Truth: negative | BERT: positive | Confidence: 0.914 | Words: 273 [LONG]
     Hulk Hogan stars as a champion wrestler (A real acting stretch...) named Rip, who is forced to defend his honor, his title and his girlfriend from a greedy corporation that wanted him to sign for their network (Because wrestling sells!) however when Rip declines, the network gets a circuit fighting championship called (and i'm totally serious) "Battle of the tough guys" who's champion Zeus (Played...

[2] Truth: positive | BERT: negative | Confidence: 0.994 | Words: 225 [LONG]
     Semper Fi! I saw "The D.I." in 1957. Two-and-a-half -years later I joined the Corps.<br /><br />Web and company got it as right as they could in '57. Boot, in '59, was more like, in fact, exactly like, the Boot Camp shown in "Full Metal Jacket" - Yes. A black recruit, in my training platoon, was called "Snowball." I was called "S

### Observations: BERT Failures

*(written after looking at the examples above — informal, honest)*

- a lot of BERT failures happen when the review has **structural complexity** —
  the sentiment isn't in the first few clauses, or a negative framing resolves
  into a positive conclusion. bert seems to latch onto the dominant keyword cluster,
  not the *arc* of the review.

- **confidence is deceptive.** bert gives 0.80–0.95 confidence on wrong predictions.
  the gap between `confidence_on_correct` vs `confidence_on_incorrect` is smaller
  than expected. this makes raw BERT confidence unsafe as a routing signal.

- mixed-sentiment reviews appear frequently in BERT failures. if a review says
  "great acting but a terrible script", bert seems to average these into whichever
  keyword cluster is stronger in its training distribution. it doesn't reason about
  clause-level sentiment.

- the `long` tag doesn't fully explain BERT failures — some short reviews fail too.
  it's more about *how* sentiment is expressed than raw length.

**H2** support: *partially confirmed.* brittleness is real but the axis is
local vs non-local sentiment expression, not length per se.

In [13]:
print('=== LLM FAILURE CASES (BERT was correct on these) ===')
print('i.e., cases BERT handles that the LLM cannot\n')

sample_llm_fails = only_llm_fails.sample(
    min(8, len(only_llm_fails)), random_state=SEED
)

for i, (_, row) in enumerate(sample_llm_fails.iterrows()):
    gt_str  = 'positive' if row['ground_truth'] == 1 else 'negative'
    llm_str = 'positive' if row['llm_pred']     == 1 else 'negative'
    raw_out = row['llm_label_raw']
    tag_str = '[LONG]' if row['tag_long'] else ''
    print(f'[{i+1}] Truth: {gt_str} | LLM: {llm_str} '
          f"(raw: '{raw_out}') | Words: {row['word_count']} {tag_str}")
    print(f'     {row["text"][:400]}...')
    print()

=== LLM FAILURE CASES (BERT was correct on these) ===
i.e., cases BERT handles that the LLM cannot

[1] Truth: positive | LLM: negative (raw: 'mixed') | Words: 164 [LONG]
     I chose this movie really for my husband-who works in radio broadcasting. I thought that it would be more of a movie that he would enjoy and relate too, though it was from the eighties-so it was a little dated. This movie really draws you in. At times you just want to strangle the host, Barry. At times you just want to send some of the bigots who call in to a true concentration camp. At times you ...

[2] Truth: negative | LLM: positive (raw: 'positive') | Words: 661 [LONG]
     It is said that there are some people out there who actually ADMIRE Monogram's movies. Well -- and why not? Monogram Studios lived on a kind of Cost Plus basis; cost, plus enough to pay the rent and buy a pizza and a bottle of robust muscatel every once in a while. Sure, they're cheap. But let's face it: they're coarse, fast, Philistine, 

### Observations: LLM Failures

*(informal — written while looking at actual output)*

- some LLM failures look like **over-reading** — the model finds nuance where
  there isn't any. a direct, angry negative review written in sophisticated language
  sometimes gets classified as positive. articulate writing might pattern-match to
  positive IMDb reviews in gpt-4o-mini's training distribution.

- the **200-word truncation** probably hurt on a handful of cases. reviews that build
  toward a conclusion have their payoff cut off. this is a direct consequence of the
  cost-control decision — it's a real tradeoff, not just a limitation to footnote.

- **rhetorical negation** is interesting. if a review says "i can't imagine anyone
  enjoying this", surface keywords ("enjoy") could flip the classification. even
  with broader context understanding, edge cases like this slip through.

- the LLM failure rate is lower than BERT's overall — but the failures that happen
  are harder to predict. BERT failures have clearer structural patterns; LLM failures
  feel more random. (which could mean: the label is noisy, or genuine model errors —
  probably both at 500 samples).

**H1** support: *tentatively confirmed.* LLM recovers from mixed-sentiment reviews
better than BERT. need more samples to be confident.

In [14]:
print('=== ACCURACY BREAKDOWN: LONG vs SHORT ===\n')
print(f'{"Category":<32} {"BERT":>7} {"LLM":>7} {"Samples":>8}')
print('-' * 57)

for tag_val, label in [
    (True,  f'Long  (>{LONG_THRESHOLD} words)'),
    (False, f'Short (<={LONG_THRESHOLD} words)'),
]:
    subset = results_df[results_df['tag_long'] == tag_val]
    b_acc  = subset['bert_correct'].mean()
    l_acc  = subset['llm_correct'].mean()
    print(f'  {label:<30} {b_acc:>6.1%} {l_acc:>6.1%} {len(subset):>8}')

print()
# BERT confidence by length — does it know it's struggling on long reviews?
long_conf  = results_df[ results_df['tag_long']]['bert_confidence'].mean()
short_conf = results_df[~results_df['tag_long']]['bert_confidence'].mean()
print(f'BERT avg confidence — long reviews:  {long_conf:.3f}')
print(f'BERT avg confidence — short reviews: {short_conf:.3f}')

=== ACCURACY BREAKDOWN: LONG vs SHORT ===

Category                            BERT     LLM  Samples
---------------------------------------------------------
  Long  (>40 words)               88.1%  94.8%      496
  Short (<=40 words)             100.0% 100.0%        4

BERT avg confidence — long reviews:  0.974
BERT avg confidence — short reviews: 0.999


### Observations: Long vs Short

- if H3 ("LLM handles long reviews better") holds, we'd expect a wider accuracy
  gap on the long subset. check the numbers above.

- BERT confidence on long vs short is telling. if confidence stays equally high
  on long reviews despite lower accuracy, that confirms the calibration problem:
  the model doesn't know it's likely truncating crucial context.

- most IMDb reviews are long — so "long" is the majority case here, not an edge case.
  deployment implication: if your data is long-form, BERT's truncation behavior is
  a first-class concern, not an edge case to acknowledge and move on from.

In [15]:
results_df.to_csv('data/results.csv', index=False)
print('Final results saved to data/results.csv')
print(f'Shape: {results_df.shape}')
print(f'Columns: {list(results_df.columns)}')

Final results saved to data/results.csv
Shape: (500, 17)
Columns: ['text', 'ground_truth', 'word_count', 'bert_pred', 'bert_label_raw', 'bert_confidence', 'bert_latency_s', 'bert_correct', 'llm_pred', 'llm_label_raw', 'llm_latency_s', 'llm_input_tokens', 'llm_output_tokens', 'llm_correct', 'tag_long', 'tag_mixed', 'tag_sarcasm']


## Section 6 — Key Insights

*The goal here is understanding, not presenting. These are directional, not conclusive.*

---

**1. BERT is "good enough" for clear-cut cases**

On simple, unambiguous reviews, BERT and the LLM perform similarly.
BERT runs in ~10–50ms on CPU and costs nothing beyond setup.
At scale, if your data is mostly straightforward, BERT is the right call.

---

**2. LLM has a measurable edge on structural complexity**

When sentiment isn't expressed in local keywords — it builds across the review
or requires understanding the *arc* of the text — the LLM handles it better.
BERT's failure mode looks like a weighted keyword lookup.
When the dominant surface keywords don't reflect the overall sentiment, BERT fails.

---

**3. BERT confidence is not well-calibrated**

BERT gives high confidence on wrong predictions. The gap between
`confidence_on_correct` and `confidence_on_incorrect` is smaller than expected.
If you're using BERT's confidence score as a routing signal ("escalate uncertain
cases to an LLM"), you need calibration first — temperature scaling or Platt scaling.
Raw confidence ≠ reliability.

---

**4. Input truncation has real accuracy consequences**

Truncating LLM input to 200 words hurt on cases where the review's sentiment
resolution was in the second half. This isn't just a footnote — it's a design
decision with accuracy tradeoffs. If you're deploying an LLM on long-form content,
think carefully about what you're feeding it.

---

**5. The cost tradeoff is clear — but not simple**

LLM cost ≈ $80/1M samples. BERT ≈ $0 at scale.
Whether that gap is worth the accuracy delta depends on the use case:
- **High-volume production:** BERT + fine-tuning almost always wins
- **Exploratory / low-volume / no labels:** LLM wins
- **Mixed traffic:** route by confidence → BERT for clear cases, LLM for uncertain ones

---

**6. Disagreements are the most valuable signal**

Every disagreement between BERT and LLM is worth inspecting.
The disagreement rate is itself an interesting metric — it captures cases where
the models have fundamentally different reads on the same text.
"BERT and LLM disagree" is a cheap, label-free proxy for "this case is ambiguous."

---

### What surprised me

- LLM failures are harder to predict than BERT failures. BERT failures have
  structural patterns. LLM failures feel almost random — which is either
  genuine model uncertainty, or label noise. Probably both at 500 samples.

- BERT confidence calibration is worse than I expected.

- The 200-word truncation hurt more than anticipated. I expected it to be
  a non-issue for most reviews; it wasn't.

---

*See `README.md` for the short write-up and findings.*